In [2]:
from pathlib import Path


IGNORE_DIRS = {
    ".git",
    "node_modules",
    "__pycache__",
    ".venv",
    "venv",
    "env",
    "dist",
    "build",
    ".next",
    "coverage",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".idea",
    ".vscode",
    "storage",
}

# Specific paths to ignore
IGNORE_PATHS = {
    Path("components/ui"),
    Path("hooks"),
}


CODE_EXTENSIONS = {
    # Frontend
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".html",
    # ".css",
    ".scss",
    ".sass",

    # Backend
    ".py",
    ".java",
    ".go",
    ".rs",
    ".php",
    ".rb",
    ".cs",
    ".cpp",
    ".c",
    ".h",

    # Database
    ".sql",
}


def should_ignore(file_path, root_folder):
    """
    Check whether a file should be ignored.
    """

    # 1. Ignore directories such as node_modules, .git, __pycache__
    if any(part in IGNORE_DIRS for part in file_path.parts):
        return True

    # 2. Check project-relative path
    relative_path = file_path.relative_to(root_folder)

    # Ignore exact paths such as components/ui
    for ignore_path in IGNORE_PATHS:
        if (
            relative_path == ignore_path
            or ignore_path in relative_path.parents
        ):
            return True

    return False


def read_code_folder(folder_path):
    folder = Path(folder_path).resolve()

    if not folder.exists():
        raise FileNotFoundError(
            f"Folder not found: {folder}"
        )

    all_code = []

    for file_path in folder.rglob("*"):

        # Ignore unwanted files/directories
        if should_ignore(file_path, folder):
            continue

        # Only files
        if not file_path.is_file():
            continue

        # Only source-code files
        if file_path.suffix.lower() not in CODE_EXTENSIONS:
            continue

        try:
            content = file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            relative_path = file_path.relative_to(folder)

            all_code.append({
                "file": str(relative_path),
                "content": content
            })

        except Exception as e:
            print(f"Could not read {file_path}: {e}")

    return all_code


# Frontend
frontend_code = read_code_folder("frontend")

# Backend
backend_code = read_code_folder("backend")

# Combine
all_code = frontend_code + backend_code


for item in all_code:
    print("=" * 80)
    print(f"FILE: {item['file']}")
    print("=" * 80)
    print(item["content"])

FILE: next-env.d.ts
/// <reference types="next" />
/// <reference types="next/image-types/global" />
import "./.next/dev/types/routes.d.ts";
import "./.next/dev/types/root-params.d.ts";

// NOTE: This file should not be edited
// see https://nextjs.org/docs/app/api-reference/config/typescript for more information.

FILE: next.config.ts
import type { NextConfig } from "next";

const nextConfig: NextConfig = {
  /* config options here */
  allowedDevOrigins: ['192.168.1.31']
};

export default nextConfig;

FILE: proxy.ts
import { NextRequest, NextResponse } from "next/server";

const API_URL =
    process.env.NEXT_PUBLIC_API_URL ?? "http://localhost:8000/api";

const ACCESS_COOKIE = "access_token";
const REFRESH_COOKIE = "refresh_token";
const PUBLIC_PATHS = ["/login", "/register"];

const isPublicPath = (pathname: string) =>
    PUBLIC_PATHS.some(
        (path) => pathname === path || pathname.startsWith(`${path}/`)
    );

async function refreshSession(refreshToken: string): Promise<s